# Intent Model Evaluation Metrics (Colab)

Use this notebook to generate evaluation metrics and visual proof (summary table, per-intent F1, confusion matrix heatmap).

In [ ]:
!pip -q install transformers datasets accelerate torch pandas matplotlib seaborn

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def find_repo_root() -> Path:
    candidates = [
        Path('/content/StylesenseSL'),
        Path('/content'),
        Path.cwd(),
    ]
    for c in candidates:
        if (c / 'backend' / 'src' / 'services' / 'agentic_ai').exists():
            return c
    raise RuntimeError('Could not find repo root. Clone repo under /content first.')

REPO_ROOT = find_repo_root()
BACKEND_ROOT = REPO_ROOT / 'backend'
MODEL_DIR = BACKEND_ROOT / 'src' / 'services' / 'agentic_ai' / 'agents' / 'models' / 'intent_distilbert'
DATA_CSV = BACKEND_ROOT / 'src' / 'services' / 'agentic_ai' / 'data' / 'intent' / 'intent_dataset_8400.csv'
REPORT_DIR = MODEL_DIR / 'evaluation_reports'

print('REPO_ROOT:', REPO_ROOT)
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
cmd = [
    'python', '-m', 'src.services.agentic_ai.scripts.evaluate_intent_distilbert',
    '--data', str(DATA_CSV.relative_to(BACKEND_ROOT).as_posix()),
    '--model-dir', str(MODEL_DIR.relative_to(BACKEND_ROOT).as_posix())
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)

In [ ]:
summary = json.loads((REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
per_intent = pd.read_csv(REPORT_DIR / 'per_intent_metrics.csv')
conf_mat = pd.read_csv(REPORT_DIR / 'confusion_matrix.csv', index_col=0)

print('Summary metrics:')
print(json.dumps(summary, indent=2))

per_intent.head()

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=per_intent.sort_values('f1', ascending=False), x='intent', y='f1', color='#2563eb')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.title('Per-Intent F1 Score')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(conf_mat, cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: save proof artifacts as images for panel/demo slides
proof_dir = REPORT_DIR / 'proof_exports'
proof_dir.mkdir(parents=True, exist_ok=True)

fig1 = plt.figure(figsize=(12, 5))
sns.barplot(data=per_intent.sort_values('f1', ascending=False), x='intent', y='f1', color='#2563eb')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.title('Per-Intent F1 Score')
plt.tight_layout()
fig1.savefig(proof_dir / 'per_intent_f1.png', dpi=180)
plt.close(fig1)

fig2 = plt.figure(figsize=(12, 10))
sns.heatmap(conf_mat, cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
fig2.savefig(proof_dir / 'confusion_matrix.png', dpi=180)
plt.close(fig2)

print('Saved proof images in:', proof_dir)